# FDA Capstone master control notebook

Run Sections 00–21 in order, pausing at Azure job boundaries. The active comparison uses retrospective inspection features and a shared FEI holdout. Azure action switches are local to each section.

## 00 Set workspace names and action switches

Confirm the workspace, cluster, and run folder. Rerun this cell only after a kernel restart or a change to shared settings. Actions are enabled in their own cells.

In [1]:
workspace = {
    "subscription_id": "cf903865-71df-48c8-9cb3-1938981c8c3a",
    "resource_group_name": "rg-udacity-azureml",
    "workspace_name": "udacity-project",
}
cluster_name = "cpu-cluster"
DATA_VERSION = "5"

OUTPUT_RUN = "retrospective-v1"
AUTO_MODEL_VERSION = "5"
SWEEP_MODEL_VERSION = "5"

ENDPOINT = "fda-capstone-yoots-01"
DEPLOYMENT = "blue"
SKU = "Standard_DS1_v2"
OWNER = "fda-capstone"

## 01 Verify repository and kernel

Run after Cell 00. Reuse the Project 2 SDK v2 kernel and confirm the printed Capstone root and CPU cluster.

In [2]:
from pathlib import Path
import json
import sys

import mltable
import pandas as pd
from azure.ai.ml import MLClient, Input
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if p.name in ("azure-ml-capstone-sdk-v2", "Capstone")
            and (p / "data/raw").is_dir())
client = MLClient(DefaultAzureCredential(), **workspace)
cluster = client.compute.get(cluster_name)
sys.path.insert(0, str(root))
output_dir = root / "outputs" / OUTPUT_RUN
output_dir.mkdir(parents=True, exist_ok=True)
from hashlib import sha256 as digest
tracked = [root / path for path in ("src/prepare_data.py", "src/train.py", "pipeline/evaluation.yml",
                                  "environment/conda_sweep.yaml", "pipeline/automl_inputs.py")]
recipe = digest(b"".join(digest(path.read_bytes()).digest() for path in tracked)).hexdigest()[:12]
output_dir = output_dir / f"data-{DATA_VERSION}-{recipe}"
output_dir.mkdir(parents=True, exist_ok=True)

print(root, sys.executable, cluster.name, sep="\n")

from uuid import uuid4


def save_job_id(path, job_id):
    temporary = path.with_name(f".{path.name}.{uuid4().hex}.tmp")
    temporary.write_text(job_id, encoding="utf-8")
    temporary.replace(path)


def reuse_or_submit(stage, experiment, path, allow_submit, build_job,
                    known_ids=(), recovered_id=""):
    """Prefer a completed run, then an active run, for this workflow."""
    scope = {
        "capstone_stage": stage,
        "capstone_output_run": OUTPUT_RUN,
        "capstone_data_version": str(DATA_VERSION),
    }
    saved_id = path.read_text(encoding="utf-8").strip() if path.exists() else ""
    ids = dict.fromkeys(filter(None, (recovered_id.strip(), *known_ids, saved_id)))
    candidates = []
    for job_id in ids:
        job = client.jobs.get(job_id)
        if job.experiment_name != experiment:
            raise RuntimeError(f"{job_id} belongs to {job.experiment_name}, expected {experiment}.")
        if any(key in (job.tags or {}) and job.tags[key] != value
               for key, value in scope.items()):
            raise RuntimeError(f"{job_id} belongs to a different Capstone workflow.")
        candidates.append(job)

    completed = next((job for job in candidates if job.status == "Completed"), None)
    if completed is None:
        # New submissions carry workflow tags, so they can be recovered if a local ID is lost.
        candidates.extend(
            job for job in client.jobs.list()
            if job.name not in ids and job.experiment_name == experiment
            and all((job.tags or {}).get(key) == value for key, value in scope.items())
        )
        completed = next((job for job in candidates if job.status == "Completed"), None)
    if completed is not None:
        save_job_id(path, completed.name)
        print(f"Reusing {stage}: {completed.name} ({completed.status})")
        return completed.name

    active = next((job for job in candidates if job.status != "Failed"), None)
    if active is not None:
        save_job_id(path, active.name)
        print(f"Keeping {stage}: {active.name} ({active.status}). No new job submitted.")
        return active.name

    if not allow_submit:
        print(f"No completed or active {stage} job. Enable its submission switch to run it.")
        return candidates[0].name if candidates else None

    for failed in candidates:
        archive = path.with_name(f"{path.stem}_{failed.name}_failed.txt")
        if not archive.exists():
            save_job_id(archive, failed.name)
    job = build_job()
    job.tags = {**(job.tags or {}), **scope}
    submitted = client.jobs.create_or_update(job)
    save_job_id(path, submitted.name)
    print(f"Submitted {stage}: {submitted.name} ({submitted.status})")
    return submitted.name

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-capstone-sdk-v2
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/.venv/bin/python
cpu-cluster


## 02 Verify the staged FDA workbooks
Upload both workbooks to data/raw in Studio Files before running.

Expected: The filename, byte size and checksum appear.

Recovery: If using the same snapshot, compare the SHA-256 with the manifest in the Part 1 appendix notes.

In [3]:
import importlib
import src.prepare_data as prepare_data

importlib.invalidate_caches()
importlib.reload(prepare_data)

print(prepare_data.__file__)
print("CAPSTONE_OUT:", getattr(prepare_data, "CAPSTONE_OUT", "MISSING"))

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-capstone-sdk-v2/src/prepare_data.py
CAPSTONE_OUT: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-capstone-sdk-v2/data/prepared-retrospective


In [4]:
from src.prepare_data import RAW, CITATIONS, CAPSTONE_OUT, sha256
source_hashes = {key: sha256(path) for key, path in (("inspections", RAW), ("citations", CITATIONS))}
print(source_hashes)

{'inspections': 'd3334dcde8a322af8b8dfdd9fa1b5a5203c2ea3a66f97acfdc91844b481855bb', 'citations': '2a8a3f8e7a23d927ebbc1c94e2ba85d91aec66b27c798d96f8a56b289d863864'}


## 03 Prepare and inspect the FEI-grouped splits

Generate the 70% retrospective cohort and FEI-disjoint splits. Both branches use the same selected numeric columns.

In [5]:
PREPARE_DATA = False  # Enable once when the retrospective split files are missing.
from src.prepare_data import prepare_capstone
from src.train import select_shared_features

prepared = CAPSTONE_OUT / DATA_VERSION / recipe
manifest_path = prepared / "manifest.json"
if PREPARE_DATA and not manifest_path.is_file():
    prepare_capstone(RAW, CITATIONS, prepared, select_shared_features)
if not manifest_path.is_file():
    raise RuntimeError("Enable PREPARE_DATA once to generate the retrospective splits.")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest["sources"] != source_hashes:
    raise RuntimeError("The workbooks changed. Use a new data version.")
print(manifest["split_policy"])
display(pd.DataFrame(manifest["splits"]).T)
print("Shared predictors:", manifest["feature_names"])

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/.venv/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/.venv/lib/python3.10/site-packages/mlflow/pyfunc/ut

,rows,facilities,positive
train,111860,62907,4584
validation,28189,15727,1148
test,46498,26212,1917


Shared predictors: {'feature_00': 'country_area_China', 'feature_01': 'first_additional_details_FSVP', 'feature_02': 'first_product_types_Food/Cosmetics', 'feature_03': 'first_project_areas_Drug Quality Assurance', 'feature_04': 'first_project_areas_Tobacco Post-Market Activities', 'feature_05': 'first_project_areas_Unapproved And Misbranded Drugs', 'feature_06': 'previous_inspection_citations', 'feature_07': 'previous_oai_recent', 'feature_08': 'prior_non_compliant_rate', 'feature_09': 'sum_posted_citations'}


## 04 Read local MLTables
Run after 03 to exercise the MLTable definitions before uploading.

Expected: Three tables with eleven features and one binary target each.

Recovery: Each split folder must contain the extensionless MLTable file and rows.parquet.

In [6]:
import importlib
import pipeline.data_assets as data_assets

importlib.invalidate_caches()
importlib.reload(data_assets)

print("Schema:", manifest["schema_version"])
print("Module:", data_assets.__file__)

Schema: fda-retrospective-v1
Module: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-capstone-sdk-v2/pipeline/data_assets.py


In [7]:
from pipeline.data_assets import load_local, register_splits, verify_splits
local = load_local(prepared, manifest)
for split, frame in local.items():
    print(split, frame.shape)

train (111860, 11)
validation (28189, 11)
test (46498, 11)


## 05 Retrieve or register exact asset versions

Existing versioned assets are retrieved with the switch off. Enable it here only to register missing assets.

In [8]:
REGISTER_DATA = True  # True only to register missing assets.

assets = register_splits(client, prepared, local, DATA_VERSION, REGISTER_DATA, prefix="fda-retrospective")
for split, asset in assets.items():
    print(split, asset.name, asset.version)

train fda-retrospective-train 5
validation fda-retrospective-validation 5
test fda-retrospective-test 5


## 06 Verify Azure readback and save the Part 1 handoff
Run after all three assets resolve in 05. Open Studio Data in a second tab.

Expected: Three Azure readbacks match the local tables and the handoff JSON is written.

Recovery: Check datastore permissions for cloud read errors. Capture the asset list and training preview after success.

In [9]:
references = verify_splits(client, assets, local, manifest["columns"])
part1 = {
    "part1_status": "COMPLETED",
    "workspace_name": workspace["workspace_name"],
    "compute_cluster": cluster_name,
    "target": "is_non_compliant",
    "assets": references,
    "data_manifest": manifest,
}
path = output_dir / "part1_workflow_state.json"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(part1, indent=2), encoding="utf-8")
print(references, path)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to i

{'train': 'azureml:fda-retrospective-train:5', 'validation': 'azureml:fda-retrospective-validation:5', 'test': 'azureml:fda-retrospective-test:5'} /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-capstone-sdk-v2/outputs/retrospective-v1/data-5-2e8ab824b705/part1_workflow_state.json


## 07 Restore the Part 1 handoff and MLflow tracking
After a restart, rerun 00–01, then run this cell.

Expected: Three pinned asset references and the workspace MLflow connection are available.

Recovery: Return to 06 if the handoff is missing or workspace/cluster differs.

In [10]:
import mlflow
from mlflow.tracking import MlflowClient

state = json.loads(
    (output_dir / "part1_workflow_state.json").read_text(encoding="utf-8")
)
assert state["part1_status"] == "COMPLETED"
assert state["workspace_name"] == workspace["workspace_name"]
assert state["compute_cluster"] == cluster_name
refs = state["assets"]
assert set(refs) == {"train", "validation", "test"}
if any(ref.rsplit(":", 1)[-1] != DATA_VERSION for ref in refs.values()):
    raise RuntimeError(f"Cell 00 expects data version {DATA_VERSION}, but the saved handoff contains {refs}.")
target = state["target"]
features = [c for c in state["data_manifest"]["columns"] if c != target]
from src.prepare_data import CAPSTONE_SCHEMA
assert state["data_manifest"]["schema_version"] == CAPSTONE_SCHEMA

def get_asset(split):
    scheme, name, version = refs[split].split(":")
    assert scheme == "azureml"
    return client.data.get(name=name, version=version)

for split in ("train", "validation", "test"):
    asset = get_asset(split)
    assert asset.type == AssetTypes.MLTABLE
    print(split, refs[split])

ws = client.workspaces.get(workspace["workspace_name"])
mlflow.set_tracking_uri(ws.mlflow_tracking_uri)
mlflow.set_registry_uri(ws.mlflow_tracking_uri)
tracking = MlflowClient()

train azureml:fda-retrospective-train:5
validation azureml:fda-retrospective-validation:5
test azureml:fda-retrospective-test:5


## 08 Submit or recover the AutoML parent

Train AutoML on the same retrospective training and validation assets as the custom branch.

In [11]:
SUBMIT_AUTO = False
RECOVER_AUTO_JOB_ID = ""

from azure.ai.ml import automl
from azure.ai.ml.automl import ClassificationModels

auto_experiment = f"capstone-fda-retro-automl-{DATA_VERSION}-{recipe}"

def build_auto_job():
    job = automl.classification(
        compute=cluster_name, experiment_name=auto_experiment,
        training_data=Input(type=AssetTypes.MLTABLE, path=refs["train"]),
        validation_data=Input(type=AssetTypes.MLTABLE, path=refs["validation"]),
        target_column_name=target, primary_metric="norm_macro_recall",
    )
    job.set_limits(timeout_minutes=90, trial_timeout_minutes=25, max_trials=4, max_concurrent_trials=1)
    job.set_training(
        allowed_training_algorithms=[ClassificationModels.LIGHT_GBM, ClassificationModels.LOGISTIC_REGRESSION],
        enable_stack_ensemble=False, enable_vote_ensemble=False,
    )
    return job

AUTO_JOB_ID = reuse_or_submit(
    "automl", auto_experiment, output_dir / "part2_automl_job_id.txt",
    SUBMIT_AUTO, build_auto_job, recovered_id=RECOVER_AUTO_JOB_ID,
)

Reusing automl: busy_bee_d9jd38kdbb (Completed)


## 09 Inspect and register the AutoML winner

Leave REGISTER_AUTO_MODEL=False to verify the existing model record. For a newly completed run, enable it here only after checking the winning child and artifact.

In [12]:
import importlib
import inspect
import src.model_registry as model_registry

importlib.invalidate_caches()
importlib.reload(model_registry)
print(inspect.signature(model_registry.register_candidate))

(client, name, version, source, child_id, allow_create=False, *, schema_version='fda-source-v3')


In [13]:
REGISTER_AUTO_MODEL = False

from src.model_registry import register_candidate
from azure.core.exceptions import ResourceNotFoundError

if not AUTO_JOB_ID or client.jobs.get(AUTO_JOB_ID).status != "Completed":
    print("Run Section 08 and wait for AutoML to complete.")
else:
    parent = tracking.get_run(AUTO_JOB_ID)
    child = parent.data.tags.get("automl_best_child_run_id")
    if not child:
        raise RuntimeError("Verify the winning AutoML child in Azure Studio.")
    trial_run = tracking.get_run(child)
    print(child, trial_run.data.metrics, trial_run.data.params)
    name = "capstone-fda-retrospective-automl"
    try:
        existing = client.models.get(name=name, version=AUTO_MODEL_VERSION)
    except ResourceNotFoundError:
        existing = None
    if existing is not None or REGISTER_AUTO_MODEL:
        model = register_candidate(
            client, name, AUTO_MODEL_VERSION,
            f"azureml://jobs/{child}/outputs/artifacts/paths/outputs/mlflow-model/",
            child, allow_create=REGISTER_AUTO_MODEL, schema_version=CAPSTONE_SCHEMA,
        )
        record = dict(name=model.name, version=model.version, child_job=child, parent_job=AUTO_JOB_ID,
                      training_assets=refs, parameters=trial_run.data.params,
                      validation_accuracy=trial_run.data.metrics.get("accuracy"))
        (output_dir / "part2_automl_model.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    else:
        print("Enable REGISTER_AUTO_MODEL after reviewing the winner.")

busy_bee_d9jd38kdbb_1 {'balanced_accuracy': 0.7572879394523764, 'f1_score_weighted': 0.8405375235429996, 'f1_score_macro': 0.5385321017557798, 'weighted_accuracy': 0.7740970593177648, 'precision_score_weighted': 0.9507868268643078, 'precision_score_micro': 0.772783709957785, 'recall_score_macro': 0.7572879394523764, 'average_precision_score_micro': 0.7775451861626415, 'norm_macro_recall': 0.5145758789047528, 'log_loss': 0.5493187720017922, 'precision_score_macro': 0.5540718409751191, 'AUC_weighted': 0.8300693410844573, 'average_precision_score_weighted': 0.9561735200163741, 'average_precision_score_macro': 0.5799579634082418, 'recall_score_micro': 0.772783709957785, 'AUC_macro': 0.8300693410844573, 'accuracy': 0.772783709957785, 'recall_score_weighted': 0.772783709957785, 'AUC_micro': 0.7889628981408077, 'f1_score_micro': 0.772783709957785, 'matthews_correlation': 0.2358985591044164} {}


## 10 Build the command job and sweep definitions

Load the repository command definition. The smoke checks the same training and artifact contract used by the sweep.

In [14]:
import yaml
from azure.ai.ml import Output, command
from azure.ai.ml.entities import Environment
from azure.ai.ml.sweep import BanditPolicy, Choice, Uniform

definition = root / "pipeline/evaluation.yml"
spec = yaml.safe_load(definition.read_text(encoding="utf-8"))
environment = spec["environment"]
conda_file = definition.parent / environment["conda_file"]
smoke_experiment = f"capstone-fda-retro-smoke-{DATA_VERSION}-{recipe}"
sweep_experiment = f"capstone-fda-retro-sweep-{DATA_VERSION}-{recipe}"
common_inputs = dict(spec["inputs"])
for split in ("train", "validation"):
    common_inputs[split] = Input(type=AssetTypes.MLTABLE, path=refs[split], mode="download")
trial = command(
    code=str((definition.parent / spec["code"]).resolve()), command=spec["command"],
    inputs=common_inputs, outputs={name: Output(**settings) for name, settings in spec["outputs"].items()},
    environment=Environment(image=environment["image"], conda_file=str(conda_file.resolve())),
    compute=cluster_name, experiment_name=smoke_experiment,
)
sweep = trial(
    depth=Choice(values=[2, 4, 6, 9, 13]), trees=Choice(values=[50, 100, 180]),
    rate=Uniform(min_value=.025, max_value=.2), smote=Choice(values=[0., .1, .2, .3, .5, .7, 1.]),
    weight=Choice(values=[1., 2., 4., 8., 16.]),
    row_fraction=Uniform(min_value=.7, max_value=1.),
    column_fraction=Uniform(min_value=.7, max_value=1.),
).sweep(compute=cluster_name, sampling_algorithm="random", primary_metric="val_oai_recall",
        goal="maximize", early_termination_policy=BanditPolicy(evaluation_interval=1, delay_evaluation=3,
                                                                slack_amount=.10))
sweep.experiment_name = sweep_experiment
sweep.set_limits(max_total_trials=10, max_concurrent_trials=1, timeout=14400, trial_timeout=1800)
print("Prepared smoke and sweep on:", refs)

Prepared smoke and sweep on: {'train': 'azureml:fda-retrospective-train:5', 'validation': 'azureml:fda-retrospective-validation:5', 'test': 'azureml:fda-retrospective-test:5'}


## 11 Submit or recover the smoke trial

A completed smoke job is reused automatically. Enable SUBMIT_SMOKE for a first run or a confirmed failure. Run Section 10 before submitting. If the original ID file is missing, paste the verified Studio job ID into RECOVER_SMOKE_JOB_ID.

In [15]:
SUBMIT_SMOKE = False  # Allow the first submission or a failed-job retry.
RECOVER_SMOKE_JOB_ID = ""  # Optional verified smoke ID from Studio.

SMOKE_JOB_ID = reuse_or_submit(
    "smoke", smoke_experiment, output_dir / "part2_smoke_job_id.txt",
    SUBMIT_SMOKE, lambda: trial, recovered_id=RECOVER_SMOKE_JOB_ID,
)

Reusing smoke: modest_plow_gdgt7tx8l8 (Completed)


## 12 Submit or recover the sweep

Reuse a completed sweep even if the smoke job ID is unavailable. A new sweep requires a completed smoke job and SUBMIT_SWEEP=True. If the original sweep ID file is missing, paste the verified Studio parent ID into RECOVER_SWEEP_JOB_ID.

In [16]:
SUBMIT_SWEEP = False  # Allow the first submission or a failed-job retry.
RECOVER_SWEEP_JOB_ID = ""  # Optional verified sweep parent ID from Studio.

sweep_record_path = output_dir / "part2_sweep_model.json"
sweep_record = json.loads(sweep_record_path.read_text(encoding="utf-8")) if sweep_record_path.is_file() else {}


def build_sweep_job():
    smoke_file = output_dir / "part2_smoke_job_id.txt"
    smoke_id = smoke_file.read_text(encoding="utf-8").strip() if smoke_file.exists() else ""
    if not smoke_id or client.jobs.get(smoke_id).status != "Completed":
        raise RuntimeError("A new sweep requires a completed smoke job. Check Section 11.")
    return sweep


SWEEP_JOB_ID = reuse_or_submit(
    "sweep", sweep_experiment, output_dir / "part2_sweep_job_id.txt",
    SUBMIT_SWEEP, build_sweep_job, (sweep_record.get("parent_job"),),
    RECOVER_SWEEP_JOB_ID,
)

Reusing sweep: neat_bird_d9zqr3f6yv (Completed)


## 13 Review sweep trials and register its winner

Wait for the sweep to complete. Enable REGISTER_SWEEP_MODEL here only after inspecting the winning child and model artifact.

In [17]:
REGISTER_SWEEP_MODEL = True

from src.model_registry import register_candidate

if not SWEEP_JOB_ID or client.jobs.get(SWEEP_JOB_ID).status != "Completed":
    print("Wait for the sweep from Section 12 to complete.")
else:
    trials = []
    for child in client.jobs.list(parent_job_name=SWEEP_JOB_ID):
        if child.status == "Completed":
            run = tracking.get_run(child.name)
            row = {key: run.data.metrics.get(f"final_val_{key}") for key in
                   ("recall", "precision", "roc_auc", "accuracy")}
            if all(value is not None for value in row.values()):
                trials.append(dict(child_job=child.name, parameters=run.data.params, **row))
    if not trials:
        raise RuntimeError("No sweep child logged final validation metrics.")
    ranked = sorted(trials, key=lambda row: (
        row["precision"] > .12, row["recall"], row["precision"], row["roc_auc"],
    ), reverse=True)
    display(pd.DataFrame(ranked))
    winner_trial = ranked[0]
    child = winner_trial["child_job"]
    name = "capstone-fda-retrospective-xgb"
    try:
        existing = client.models.get(name=name, version=SWEEP_MODEL_VERSION)
    except ResourceNotFoundError:
        existing = None
    if existing is not None or REGISTER_SWEEP_MODEL:
        model = register_candidate(
            client, name, SWEEP_MODEL_VERSION,
            f"azureml://jobs/{child}/outputs/model/paths/mlflow-model/",
            child, allow_create=REGISTER_SWEEP_MODEL, schema_version=CAPSTONE_SCHEMA,
        )
        record = dict(name=model.name, version=model.version, parent_job=SWEEP_JOB_ID,
                      training_assets=refs, **winner_trial)
        (output_dir / "part2_sweep_model.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    else:
        print("Enable REGISTER_SWEEP_MODEL after reviewing the winner.")

,child_job,parameters,recall,precision,roc_auc,accuracy
0,neat_bird_d9zqr3f6yv_0,"{'max_depth': '2', 'n_estimators': '180', 'lea...",0.706446,0.135009,0.840070,0.803718
1,neat_bird_d9zqr3f6yv_6,"{'max_depth': '2', 'n_estimators': '100', 'lea...",0.704704,0.134542,0.838667,0.803363
2,neat_bird_d9zqr3f6yv_3,"{'max_depth': '4', 'n_estimators': '50', 'lear...",0.697735,0.135350,0.839444,0.806166
3,neat_bird_d9zqr3f6yv_23,"{'max_depth': '4', 'n_estimators': '100', 'lea...",0.695122,0.134343,0.838167,0.805172
4,neat_bird_d9zqr3f6yv_15,"{'max_depth': '4', 'n_estimators': '100', 'lea...",0.681185,0.133220,0.838501,0.806520
5,neat_bird_d9zqr3f6yv_21,"{'max_depth': '4', 'n_estimators': '100', 'lea...",0.679443,0.134136,0.838214,0.808329
6,neat_bird_d9zqr3f6yv_8,"{'max_depth': '4', 'n_estimators': '100', 'lea...",0.673345,0.133783,0.836609,0.809145
7,neat_bird_d9zqr3f6yv_19,"{'max_depth': '4', 'n_estimators': '180', 'lea...",0.673345,0.133529,0.836847,0.808755
8,neat_bird_d9zqr3f6yv_10,"{'max_depth': '9', 'n_estimators': '50', 'lear...",0.664634,0.132396,0.831227,0.808968
9,neat_bird_d9zqr3f6yv_11,"{'max_depth': '9', 'n_estimators': '50', 'lear...",0.663763,0.133031,0.833028,0.810139


## 14 Evaluate validation and holdout

Score both registered branches on the same holdout. The first successful run also computes the original benchmark and caches it.

In [18]:
import importlib
import pipeline.evaluate_automl as evaluator
import src.train as training

importlib.invalidate_caches()
importlib.reload(evaluator)
importlib.reload(training)

print("predict_candidate:", hasattr(evaluator, "predict_candidate"))
print("reference_results:", hasattr(training, "reference_results"))

/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/.venv/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-bankmarketing-sdk-v2/.venv/lib/python3.10/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [19]:
from pipeline.evaluate_automl import predict_candidate
from src.train import metrics, reference_results

record_paths = {branch: output_dir / f"part2_{branch}_model.json" for branch in ("automl", "sweep")}
comparison = None
if not all(path.is_file() for path in record_paths.values()):
    print("Register both branch winners before comparing them.")
else:
    test_asset = get_asset("test")
    table = mltable.load(f"azureml:/{test_asset.id}", ml_client=client).to_pandas_dataframe()
    assert list(table.columns) == [*features, target]
    X, y = table[features], table[target]
    records = {branch: json.loads(path.read_text(encoding="utf-8")) for branch, path in record_paths.items()}
    comparison = {}
    for branch, record in records.items():
        assert record["training_assets"] == refs, branch
        uri = f"models:/{record['name']}/{record['version']}"
        labels, scores = predict_candidate(uri, X, threshold_artifact=branch == "sweep")
        comparison[branch] = {**record, **metrics(y, [int(label) for label in labels], scores)}

    reference_file = output_dir / "reference_comparison.json"
    if not reference_file.is_file():
        reference_file.write_text(json.dumps(reference_results(RAW), indent=2), encoding="utf-8")
    reference = json.loads(reference_file.read_text(encoding="utf-8"))
    print("Original retrospective deck and tuned export reproduction:")
    display(pd.DataFrame(reference["deck"]["model_comparison"]))
    print(reference["deck"]["tuned_xgboost_reproduction"])
    view = {"Original XGBoost recipe": reference["matched_baseline"],
            "Original with tuned threshold": reference["threshold_only"], **comparison}
    table = pd.DataFrame.from_dict(view, orient="index")
    table["missed_oai"] = [row["confusion_matrix"][1][0] for row in view.values()]
    display(table[["accuracy", "recall", "precision", "f1", "roc_auc", "average_precision",
                   "false_positives", "missed_oai", "confusion_matrix"]])
    (output_dir / "part2_holdout_comparison.json").write_text(
        json.dumps(comparison, indent=2), encoding="utf-8"
    )

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Original retrospective deck and tuned export reproduction:


,model,recall,precision,accuracy,f1,roc_auc,average_precision,false_positives,confusion_matrix,deck_metrics,delta_from_deck
0,Logistic Regression,0.546154,0.149516,0.849709,0.234762,0.822228,0.148164,6058,"[[38189, 6058], [885, 1065]]","{'accuracy': 0.75, 'roc_auc': 0.8, 'precision'...","{'accuracy': 0.09970885555339093, 'roc_auc': 0..."
1,Decision Tree,0.751795,0.117685,0.751607,0.203512,0.816068,0.173079,10991,"[[33256, 10991], [484, 1466]]","{'accuracy': 0.85, 'roc_auc': 0.82, 'precision...","{'accuracy': -0.09839275277615422, 'roc_auc': ..."
2,Random Forest,0.753846,0.117817,0.751347,0.203785,0.817190,0.173883,11007,"[[33240, 11007], [480, 1470]]","{'accuracy': 0.74, 'roc_auc': 0.81, 'precision...","{'accuracy': 0.011347490096759505, 'roc_auc': ..."
3,XGBoost,0.764103,0.118536,0.750200,0.205234,0.827166,0.179824,11080,"[[33167, 11080], [460, 1490]]","{'accuracy': 0.74, 'roc_auc': 0.83, 'precision...","{'accuracy': 0.01020022945212895, 'roc_auc': -..."


{'best_parameters': {'colsample_bytree': 0.664651484177515, 'learning_rate': 0.08897175659930932, 'max_depth': 8, 'n_estimators': 97, 'subsample': 0.7991760293229221}, 'cv_recall': 0.8141637048432445, 'test': {'recall': 0.8046153846153846, 'precision': 0.11529135131163201, 'accuracy': 0.7311297270385523, 'f1': 0.2016839128478694, 'roc_auc': 0.8272077318873712, 'average_precision': 0.18299415472915562, 'false_positives': 12040, 'confusion_matrix': [[32207, 12040], [381, 1569]]}, 'original_export': {'recall': 0.7569230769230769, 'precision': 0.11443634672042177, 'roc_auc': 0.8178517911977807}, 'parameter_grid_seed': 20, 'protocol': '25 recall-scored candidates, 5 folds after training SMOTE, historical reproduction only.'}


,accuracy,recall,precision,f1,roc_auc,average_precision,false_positives,missed_oai,confusion_matrix
Original XGBoost recipe,0.735236,0.773083,0.110945,0.194043,0.812735,0.157129,11876,435,"[[32705, 11876], [435, 1482]]"
Original with tuned threshold,0.822638,0.575378,0.129217,0.211040,0.812735,0.157129,7433,814,"[[37148, 7433], [814, 1103]]"
automl,0.772248,0.715180,0.120105,0.205671,0.815443,0.149354,10044,546,"[[34537, 10044], [546, 1371]]"
sweep,0.801776,0.673970,0.130716,0.218964,0.825994,0.190536,8592,625,"[[35989, 8592], [625, 1292]]"


## 15 Compare and record the deployment winner

Show the recall, precision and ROC AUC targets separately from the accuracy-based deployment decision required by the task checklist.

In [20]:
if comparison is None:
    print("Complete Section 14 before selecting a deployment.")
else:
    branch = max(comparison, key=lambda name: (comparison[name]["accuracy"], comparison[name]["recall"]))
    chosen = comparison[branch]
    targets_met = [name for name, row in comparison.items()
                   if row["recall"] > .78 and row["precision"] > .12 and row["roc_auc"] > .83]
    print("Deck targets met by:", targets_met or "neither candidate")
    print("Udacity deployment winner by holdout accuracy:", branch, chosen["name"], chosen["version"])
    handoff = dict(
        part2_status="COMPLETED", workspace_name=workspace["workspace_name"],
        data_assets=refs, features=features, target=target, candidates=comparison,
        winner=dict(branch=branch, **{k: chosen[k] for k in ("name", "version", "child_job")}),
        targets_met=targets_met,
    )
    (output_dir / "part2_workflow_state.json").write_text(json.dumps(handoff, indent=2), encoding="utf-8")

Deck targets met by: neither candidate
Udacity deployment winner by holdout accuracy: sweep capstone-fda-retrospective-xgb 5


## 16 Restore the selected model and verify the signature
After restart run 00–01. Use the Part 2 winner and inspect signature inputs.

Expected: Verified lineage and environment/inference-requirements.txt.

Recovery: Use the registered artifact requirements and MLmodel from Studio if registry downloads fail.

In [21]:
print("Current run:", output_dir)
for name in (
    "part2_automl_model.json",
    "part2_sweep_model.json",
    "part2_holdout_comparison.json",
    "part2_workflow_state.json",
):
    print(name, (output_dir / name).is_file())

Current run: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-capstone-sdk-v2/outputs/retrospective-v1/data-5-2e8ab824b705
part2_automl_model.json True
part2_sweep_model.json True
part2_holdout_comparison.json True
part2_workflow_state.json True


In [22]:
import json
import mlflow
from azure.ai.ml.constants import AssetTypes

state_path = output_dir / "part2_workflow_state.json"
state = json.loads(state_path.read_text(encoding="utf-8"))
assert state["part2_status"] == "COMPLETED"
assert state["workspace_name"] == workspace["workspace_name"]
winner = state["winner"]
features = state["features"]
from src.prepare_data import CAPSTONE_SCHEMA
model = client.models.get(name=winner["name"], version=winner["version"])
assert (model.tags or {}).get("fda_schema") == CAPSTONE_SCHEMA
assert model.type == AssetTypes.MLFLOW_MODEL
assert (model.tags or {}).get("capstone_child_job") == winner["child_job"]
print(winner, model.id)

ws = client.workspaces.get(workspace["workspace_name"])
mlflow.set_tracking_uri(ws.mlflow_tracking_uri)
mlflow.set_registry_uri(ws.mlflow_tracking_uri)
model_uri = f"models:/{model.name}/{model.version}" 

from mlflow.models import Model as MLflowMetadata
from pathlib import Path

metadata = MLflowMetadata.load(model_uri)
assert metadata.signature and metadata.signature.inputs
model_features = metadata.signature.inputs.input_names()
assert set(model_features) == set(features)
print(metadata.signature)

deps = Path(mlflow.artifacts.download_artifacts(
    artifact_uri=f"{model_uri}/requirements.txt"
))
environment_dir = root / "environment"
environment_dir.mkdir(exist_ok=True)
(environment_dir / "inference-requirements.txt").write_text(
    deps.read_text(encoding="utf-8"), encoding="utf-8"
)
print(f"Serving: Azure ML managed MLflow, {model.name}:{model.version}")

{'branch': 'sweep', 'name': 'capstone-fda-retrospective-xgb', 'version': '5', 'child_job': 'neat_bird_d9zqr3f6yv_0'} /subscriptions/cf903865-71df-48c8-9cb3-1938981c8c3a/resourceGroups/rg-udacity-azureml/providers/Microsoft.MachineLearningServices/workspaces/udacity-project/models/capstone-fda-retrospective-xgb/versions/5


## 17 Retrieve or create the dedicated endpoint

Leave CREATE_ENDPOINT=False to inspect the current state. Enable it here only to create the dedicated endpoint.

In [23]:
ENDPOINT = "fda-capstone-yoots-01"
CREATE_ENDPOINT = True  # Look up the existing endpoint.

from azure.ai.ml.entities import (
    ManagedOnlineEndpoint, ManagedOnlineDeployment,
)
from azure.core.exceptions import ResourceNotFoundError

assert ENDPOINT != "YOUR-UNIQUE-ENDPOINT"

try:
    endpoint = client.online_endpoints.get(ENDPOINT)
except ResourceNotFoundError:
    if not CREATE_ENDPOINT:
        raise RuntimeError("Endpoint missing. Verify name and set CREATE_ENDPOINT=True.")
    endpoint = client.online_endpoints.begin_create_or_update(
        ManagedOnlineEndpoint(name=ENDPOINT, auth_mode="key",
                              tags={"purpose": OWNER})
    ).result()

assert (endpoint.tags or {}).get("purpose") == OWNER
assert endpoint.auth_mode == "key"
assert endpoint.provisioning_state == "Succeeded"
print(endpoint.name, endpoint.provisioning_state)

fda-capstone-yoots-01 EndpointProvisioningState.SUCCEEDED


In [25]:
from azure.core.exceptions import ResourceNotFoundError

try:
    model = client.models.get(name=winner["name"], version="7")
except ResourceNotFoundError as error:
    raise RuntimeError(
        "The repaired version 7 is missing from this workspace. Verify the workspace and registry in Studio."
    ) from error

tags = model.tags or {}
if (tags.get("local_http_validation") != "passed"
        or tags.get("capstone_child_job") != str(winner["child_job"])):
    raise RuntimeError("Version 7 does not match the validated training winner.")

print(f"Reusing registered model: {model.name}:{model.version}")


Reusing registered model: capstone-fda-retrospective-xgb:7


## 18 Deploy the exact version

This cell reuses the verified model version 7. Keep APPLY_DEPLOYMENT=False to inspect. Set it True once to create, update, or recover blue on the quota-compatible SKU. Wait for Verified before continuing to Section 19.

In [31]:
APPLY_DEPLOYMENT = False  # Set True once to create, update, or recover blue.
SKU = "Standard_DS1_v2"  # One instance fits the quota reported for this workspace.

from azure.ai.ml.entities import ManagedOnlineDeployment, ProbeSettings
from azure.core.exceptions import HttpResponseError, ResourceNotFoundError


def deployment_status(value):
    return str(getattr(value, "value", value)).split(".")[-1].casefold()


def deployment_model_id(value):
    return str(getattr(value, "id", value)).removeprefix("azureml:").rstrip("/").casefold()


def deployment_logs():
    for container in ("inference-server", "storage-initializer"):
        try:
            print(f"\n{container}:")
            print(client.online_deployments.get_logs(
                name=DEPLOYMENT, endpoint_name=ENDPOINT, lines=150, container_type=container,
            ))
        except HttpResponseError as error:
            print(f"{container} logs unavailable: {error}")


def deploy_verified_model():
    try:
        current = client.online_deployments.get(name=DEPLOYMENT, endpoint_name=ENDPOINT)
    except ResourceNotFoundError:
        current = None

    if current is not None:
        status = deployment_status(current.provisioning_state)
        print(f"Existing deployment: {current.name}, {status}, {current.instance_type}")
        if status == "succeeded" and deployment_model_id(current.model) == deployment_model_id(model.id):
            if (current.tags or {}).get("capstone_child_job") == str(winner["child_job"]):
                return current
        if status not in ("succeeded", "failed"):
            print("Wait for provisioning to finish. No deployment action submitted.")
            return None
    else:
        print(f"Deployment {DEPLOYMENT} is missing.")

    if not APPLY_DEPLOYMENT:
        if current is not None and status == "failed":
            deployment_logs()
        print("Set APPLY_DEPLOYMENT=True once when ready to deploy the verified model.")
        return None

    probes = dict(initial_delay=300, period=10, timeout=10, failure_threshold=30)
    desired = ManagedOnlineDeployment(
        name=DEPLOYMENT, endpoint_name=ENDPOINT, model=model.id, instance_type=SKU,
        instance_count=1,
        tags={**(current.tags or {} if current is not None else {}),
              "capstone_child_job": str(winner["child_job"])},
        environment_variables={"OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1",
                               "MKL_NUM_THREADS": "1", "NUMEXPR_NUM_THREADS": "1",
                               "WORKER_COUNT": "1"},
        liveness_probe=ProbeSettings(**probes), readiness_probe=ProbeSettings(**probes),
    )
    print(f"Applying {model.name}:{model.version} to {ENDPOINT}/{DEPLOYMENT} on {SKU}.", flush=True)
    try:
        client.online_deployments.begin_create_or_update(desired).result()
    except HttpResponseError as error:
        if "not enough quota" in str(error).casefold() or "outofquota" in str(error).casefold():
            print(f"Azure rejected the quota request for {SKU}. Check East US DSv2 available quota.")
        else:
            deployment_logs()
        raise

    current = client.online_deployments.get(name=DEPLOYMENT, endpoint_name=ENDPOINT)
    if (deployment_status(current.provisioning_state) != "succeeded"
            or deployment_model_id(current.model) != deployment_model_id(model.id)
            or (current.tags or {}).get("capstone_child_job") != str(winner["child_job"])):
        deployment_logs()
        raise RuntimeError("Azure did not verify the expected deployment and model lineage.")
    return current


# Reuse the dependency-repaired version already registered in your notebook.
model = client.models.get(name=winner["name"], version="7")
model_tags = model.tags or {}
if (model_tags.get("local_http_validation") != "passed"
        or model_tags.get("capstone_child_job") != str(winner["child_job"])):
    raise RuntimeError("Model version 7 does not match the locally validated training winner.")
endpoint = client.online_endpoints.get(ENDPOINT)
if (deployment_status(endpoint.provisioning_state) != "succeeded"
        or (endpoint.tags or {}).get("purpose") != OWNER):
    raise RuntimeError("Verify endpoint ownership and health in Section 17.")

blue = deploy_verified_model()
if blue is not None:
    print(f"Verified: {blue.name} {blue.provisioning_state} | {model.name}:{model.version}")


Existing deployment: blue, succeeded, Standard_DS1_v2
Verified: blue Succeeded | capstone-fda-retrospective-xgb:7


## 19 Route traffic and write the request

Leave UPDATE_TRAFFIC=False to inspect the current state. Enable it here only to route traffic to the verified deployment.

In [32]:
UPDATE_TRAFFIC = True  # Set True only to route traffic to the verified deployment.

endpoint = client.online_endpoints.get(ENDPOINT)
deployments = list(client.online_deployments.list(endpoint_name=ENDPOINT))
assert {item.name for item in deployments} == {DEPLOYMENT}, (
    "Check the dedicated endpoint before changing traffic."
)

if dict(endpoint.traffic or {}) != {DEPLOYMENT: 100}:
    if not UPDATE_TRAFFIC:
        raise RuntimeError("Verify the deployment, then enable UPDATE_TRAFFIC to route.")
    endpoint.traffic = {DEPLOYMENT: 100}
    endpoint = client.online_endpoints.begin_create_or_update(endpoint).result()

assert endpoint.provisioning_state == "Succeeded"
assert dict(endpoint.traffic) == {DEPLOYMENT: 100}
assert endpoint.scoring_uri
print(endpoint.name, endpoint.provisioning_state, endpoint.traffic)

from src.validate import validate_payload

train_asset = state["data_assets"]["train"]
_, asset_name, asset_version = train_asset.split(":")
asset = client.data.get(name=asset_name, version=asset_version)
examples = mltable.load(f"azureml:/{asset.id}", ml_client=client).to_pandas_dataframe()
examples = examples.loc[:, model_features].drop_duplicates().sample(n=2, random_state=20)
assert list(examples.columns) == model_features
payload = {"input_data": json.loads(examples.reset_index(drop=True).to_json(orient="split"))}
validate_payload(payload, numeric_columns=model_features)
request_path = root / "deployment/sample-request.json"
request_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
schema_path = request_path.with_name("request-schema.json")
schema_path.write_text(
    json.dumps({"numeric_columns": model_features}),
    encoding="utf-8",
)
print(request_path)

fda-capstone-yoots-01 EndpointProvisioningState.SUCCEEDED {'blue': 100}
/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-udacity-project/code/Users/ml.slayer.2019/azure-ml-capstone-sdk-v2/deployment/sample-request.json


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## 20 Verify the HTTP response in the master notebook
Run against the healthy deployed version. The same client can be run from Terminal for the screencast.

Expected: HTTP 200 and one binary label per row.

Recovery: Inspect payload/signature for 400, key-read permission for 401/403, logs and traffic for 429 or 5xx.

In [33]:
from deployment.consume import main as consume
argv = [
    "--subscription", workspace["subscription_id"],
    "--resource-group", workspace["resource_group_name"],
    "--workspace", workspace["workspace_name"], "--endpoint", ENDPOINT,
]
consume(argv)

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


HTTP 200
[
  1,
  0
]


## 21 Clean up the dedicated endpoint after recording

Leave DELETE_DEMO=False while the endpoint is needed. Enable it here only after verifying the hosted video, screenshots, and endpoint name.

In [34]:
DELETE_DEMO = False  # Enable only after collecting submission evidence.

if DELETE_DEMO:
    ep = client.online_endpoints.get(ENDPOINT)
    assert (ep.tags or {}).get("purpose") == OWNER
    names = {d.name for d in client.online_deployments.list(
        endpoint_name=ENDPOINT
    )}
    assert names == {DEPLOYMENT}, names
    client.online_endpoints.begin_delete(ENDPOINT).result()
    try:
        client.online_endpoints.get(ENDPOINT)
    except ResourceNotFoundError:
        print("Verified: the Capstone demo endpoint was deleted.")
    else:
        raise RuntimeError("The endpoint is still visible. Check Studio.")